# 债券基金久期与信用配置风格分析

> 复现华泰证券研报《基于净值数据对债券基金久期和信用配置风格进行估计的方法》(2020-08-21)

## 目录
1. [数据准备](#1-数据准备)
2. [风格估计方法](#2-风格估计方法)
3. [实证分析](#3-实证分析)
4. [滚动窗口分析](#4-滚动窗口分析)
5. [风格稳定性](#5-风格稳定性)
6. [可视化](#6-可视化)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from source.data_loader import BondDataLoader
from source.factor import BondStyleEstimator
from source.backtest import StyleBacktest
from source.plot import BondStylePlotter

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

print('环境初始化完成')

## 1. 数据准备

### 1.1 加载基金净值数据

In [ ]:
# 初始化数据加载器
loader = BondDataLoader()

# 设置参数
fund_code = '000012'  # 华夏债券A
end_date = datetime.now()
start_date = end_date - timedelta(days=365)

print(f'分析基金: {fund_code}')
print(f'分析区间: {start_date.strftime("%Y-%m-%d")} 至 {end_date.strftime("%Y-%m-%d")}')

In [ ]:
# 获取基金净值
try:
    fund_df = loader.get_fund_nav(fund_code, 
                                  start_date.strftime('%Y%m%d'),
                                  end_date.strftime('%Y%m%d'))
    print(f'获取基金数据: {len(fund_df)} 条')
except Exception as e:
    print(f'获取失败: {e}')
    print('使用模拟数据...')
    fund_df = loader._generate_mock_index_data('fund', 
                                               start_date.strftime('%Y-%m-%d'),
                                               end_date.strftime('%Y-%m-%d'))
    fund_df['nav'] = fund_df['close']
    fund_df['acc_nav'] = fund_df['close']

fund_df.head()

### 1.2 加载债券指数数据

In [ ]:
# 加载所有指数数据
index_data = loader.load_all_index_data(
    start_date.strftime('%Y-%m-%d'),
    end_date.strftime('%Y-%m-%d')
)

print(f'加载指数数量: {len(index_data)}')
for code, df in list(index_data.items())[:3]:
    print(f'  {code}: {len(df)} 条')

In [ ]:
# 查看指数信息
index_info = loader.get_index_info()
index_info.head(10)

### 1.3 数据对齐与收益率计算

In [ ]:
# 合并所有数据
all_data = {'fund': fund_df}
all_data.update(index_data)

# 对齐日期
from source.utils import align_dates
aligned_data = align_dates(all_data, date_col='date')

# 提取收益率
fund_returns = aligned_data['fund'].set_index('date')['daily_return']
index_returns = {
    code: df.set_index('date')['daily_return']
    for code, df in aligned_data.items() if code != 'fund'
}

# 构建矩阵
X = pd.DataFrame(index_returns).dropna()
y = fund_returns.loc[X.index].dropna()
X = X.loc[y.index]

print(f'对齐后数据: {len(X)} 个交易日')
print(f'因子数量: {X.shape[1]}')

## 2. 风格估计方法

### 2.1 收益率回归模型

核心模型：
$$R_{fund} = \alpha + \beta_1 R_{index1} + \beta_2 R_{index2} + ... + \beta_n R_{indexn} + \epsilon$$

约束条件：
- $\sum \beta_i = 1$ （因子暴露之和为1）
- $\beta_i \geq 0$ （非负约束）

### 2.2 久期估计

利用回归系数加权各指数久期：
$$D_{fund} = \alpha + \beta_1 D_{index1} + \beta_2 D_{index2} + ... + \beta_n D_{indexn}$$

### 2.3 信用估计

同样方法，将久期替换为信用评分：
$$C_{fund} = \alpha + \beta_1 C_{index1} + \beta_2 C_{index2} + ... + \beta_n C_{indexn}$$

## 3. 实证分析

In [ ]:
# 初始化估计器
estimator = BondStyleEstimator(index_info)

# 拟合模型
fit_result = estimator.fit(X, y, method='ols')

print(f'回归 R-squared: {fit_result["r2"]:.4f}')
print(f'截距项: {fit_result["intercept"]:.6f}')

In [ ]:
# 查看回归系数
coef_df = pd.DataFrame({
    '指数': fit_result['coef'].index,
    '系数': fit_result['coef'].values
}).sort_values('系数', ascending=False)

print('\n因子暴露（回归系数）:')
print(coef_df.to_string(index=False))

In [ ]:
# 估计久期
dur_result = estimator.estimate_duration()

print('\n=== 久期估计结果 ===')
print(f'估计久期: {dur_result["estimated_duration"]:.2f} 年')
print(f'久期风格: {dur_result["style_label"]}')

In [ ]:
# 估计信用
cred_result = estimator.estimate_credit()

print('\n=== 信用估计结果 ===')
print(f'估计信用评分: {cred_result["estimated_credit"]:.2f} 分')
print(f'信用风格: {cred_result["style_label"]}')

In [ ]:
# 风格箱定位
style_box = estimator.get_style_box()

print('\n=== 风格箱定位 ===')
print(f'风格箱: {style_box["style_box"]}')
print(f'久期: {style_box["duration"]:.2f} 年')
print(f'信用: {style_box["credit"]:.2f} 分')

## 4. 滚动窗口分析

In [ ]:
# 滚动回测
backtest = StyleBacktest(estimator)
rolling_results = backtest.run_rolling_backtest(X, y, window=60, step=20)

print(f'滚动窗口数: {len(rolling_results)}')
rolling_results.head(10)

In [ ]:
# 滚动窗口统计
print('\n滚动窗口风格统计:')
print(f"久期: 均值={rolling_results['duration'].mean():.2f}, 标准差={rolling_results['duration'].std():.2f}")
print(f"信用: 均值={rolling_results['credit'].mean():.2f}, 标准差={rolling_results['credit'].std():.2f}")
print(f"R²: 均值={rolling_results['r2'].mean():.4f}")

## 5. 风格稳定性

In [ ]:
# 计算风格稳定性
stability = backtest.calculate_style_stability()

print('\n=== 风格稳定性分析 ===')
print(f"久期稳定性: {stability['duration']['mean']:.2f} +/- {stability['duration']['std']:.2f}")
print(f"信用稳定性: {stability['credit']['mean']:.2f} +/- {stability['credit']['std']:.2f}")
print(f"久期风格变化次数: {stability['duration']['changes']}")
print(f"信用风格变化次数: {stability['credit']['changes']}")

In [ ]:
# 检测风格漂移
drifts = backtest.detect_style_drift(threshold=1.5)

if drifts:
    print(f'\n检测到 {len(drifts)} 次风格漂移:')
    for d in drifts[:5]:
        print(f"  {d['date'].strftime('%Y-%m-%d')}: {d['type']}, {d['from_value']:.2f} -> {d['to_value']:.2f}")
else:
    print('\n未检测到显著风格漂移')

## 6. 可视化

In [ ]:
plotter = BondStylePlotter()

In [ ]:
# 风格演变时序图
fig1 = plotter.plot_style_evolution(rolling_results)
plt.show()

In [ ]:
# 风格箱定位图
fig2 = plotter.plot_style_box(
    style_box['duration'], 
    style_box['credit'],
    fund_name=f'基金{fund_code}'
)
plt.show()

In [ ]:
# 因子暴露图
fig3 = plotter.plot_factor_exposure(fit_result['coef'], index_info)
plt.show()

In [ ]:
# 风格漂移检测图
fig4 = plotter.plot_style_drift(rolling_results, drifts)
plt.show()

## 总结

本Notebook完整复现了华泰证券研报《基于净值数据对债券基金久期和信用配置风格进行估计的方法》的核心算法：

1. **收益率回归**：基金收益率对债券指数收益率进行回归
2. **久期估计**：利用回归系数加权各指数久期
3. **信用估计**：利用回归系数加权各指数信用评分
4. **风格定位**：根据久期和信用评分定位风格箱
5. **稳定性分析**：滚动窗口分析风格漂移